# 3.5 Flash Attention Lab[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.5_flash_attention/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/03_attention_variants/03.5_flash_attention/lab.ipynb)**GPU Required.** This lab measures O(N²) memory growth of standard attention, implements tiled attention, and benchmarks FlashAttention backends.**Runtime:** Colab T4 or better. Select Runtime > Change runtime type > T4 GPU.

In [ ]:
# --- Setup: verify GPU and import dependencies ---import torchimport timeimport matplotlib.pyplot as pltassert torch.cuda.is_available(), "GPU required for this lab"device = torch.device("cuda")gpu_name = torch.cuda.get_device_name(0)total_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9print(f"GPU: {gpu_name} | Memory: {total_mem_gb:.1f} GB")

## Experiment 1: O(N²) Memory GrowthStandard attention materializes the full score matrix S = Q @ K^T of shape [N, N]. We allocate this matrix at increasing sequence lengths and measure memory consumption. The growth should be exactly quadratic.

In [ ]:
# --- Experiment 1: Measure O(N²) memory growth of attention scores ---seq_lengths_exp1 = [512, 1024, 2048, 4096, 8192]memory_mb_exp1 = []for seq_len in seq_lengths_exp1:    torch.cuda.empty_cache()    torch.cuda.reset_peak_memory_stats()    # Allocate N×N score matrix in FP16 (2 bytes per element)    score_matrix = torch.randn(seq_len, seq_len, dtype=torch.float16, device=device)    mem_used = torch.cuda.max_memory_allocated() / 1e6  # Convert to MB    memory_mb_exp1.append(mem_used)    # Theoretical: N² × 2 bytes    theoretical_mb = (seq_len ** 2 * 2) / 1e6    print(f"N={seq_len:>5d} | Allocated: {mem_used:>8.1f} MB | Theoretical: {theoretical_mb:>8.1f} MB")    del score_matrix# Plot quadratic growthfig_exp1, ax_exp1 = plt.subplots(figsize=(8, 5))theoretical_exp1 = [(n**2 * 2) / 1e6 for n in seq_lengths_exp1]ax_exp1.plot(seq_lengths_exp1, memory_mb_exp1, 'o-', label='Measured', linewidth=2)ax_exp1.plot(seq_lengths_exp1, theoretical_exp1, 's--', label='Theoretical N²×2B', linewidth=2)ax_exp1.set_xlabel("Sequence Length (N)")ax_exp1.set_ylabel("Memory (MB)")ax_exp1.set_title("Attention Score Matrix: O(N²) Memory Growth")ax_exp1.legend()ax_exp1.grid(True, alpha=0.3)plt.tight_layout()plt.show()

## Experiment 2: Tiled Attention (Simplified)We implement a simplified tiled attention that processes Q and K in blocks. This demonstrates the core FlashAttention idea: never materialize the full N×N matrix. We verify correctness against PyTorch's built-in SDPA.

In [ ]:
# --- Experiment 2: Simple tiled attention with online softmax ---def tiled_attention(query, key, value, block_size=64):    """Simplified tiled attention demonstrating online softmax.    Processes attention in blocks without materializing full N×N matrix."""    batch, heads, seq_len, head_dim = query.shape    output = torch.zeros_like(query)    # Running statistics for online softmax    row_max = torch.full((batch, heads, seq_len, 1), float('-inf'), device=query.device, dtype=query.dtype)    row_sum = torch.zeros((batch, heads, seq_len, 1), device=query.device, dtype=query.dtype)    scale = head_dim ** -0.5    # Process K/V in tiles    for j_start in range(0, seq_len, block_size):        j_end = min(j_start + block_size, seq_len)        # Load one tile of K and V        k_tile = key[:, :, j_start:j_end, :]   # [B, H, block, d]        v_tile = value[:, :, j_start:j_end, :]  # [B, H, block, d]        # Compute scores for this tile: [B, H, N, block]        scores_tile = torch.matmul(query, k_tile.transpose(-2, -1)) * scale        # Online softmax update        tile_max = scores_tile.max(dim=-1, keepdim=True).values        new_max = torch.maximum(row_max, tile_max)        # Rescale previous accumulations        correction = torch.exp(row_max - new_max)        output = output * correction        row_sum = row_sum * correction        # Compute current tile contribution        exp_scores = torch.exp(scores_tile - new_max)        output = output + torch.matmul(exp_scores, v_tile)        row_sum = row_sum + exp_scores.sum(dim=-1, keepdim=True)        row_max = new_max    # Normalize by total sum    output = output / row_sum    return output# Test correctness against PyTorch SDPAtorch.manual_seed(42)test_batch, test_heads, test_seq, test_dim = 2, 4, 256, 64q_test = torch.randn(test_batch, test_heads, test_seq, test_dim, device=device, dtype=torch.float32)k_test = torch.randn(test_batch, test_heads, test_seq, test_dim, device=device, dtype=torch.float32)v_test = torch.randn(test_batch, test_heads, test_seq, test_dim, device=device, dtype=torch.float32)# Reference: PyTorch SDPA (math backend for FP32 comparison)ref_output = torch.nn.functional.scaled_dot_product_attention(q_test, k_test, v_test)# Our tiled implementationtiled_output = tiled_attention(q_test, k_test, v_test, block_size=64)# Verify numerical equivalencemax_diff = (ref_output - tiled_output).abs().max().item()print(f"Max absolute difference: {max_diff:.2e}")print(f"Correctness: {'PASS ✓' if max_diff < 1e-4 else 'FAIL ✗'}  (threshold: 1e-4)")

## Experiment 3: Benchmark SDPA BackendsPyTorch's `scaled_dot_product_attention` supports three backends: `flash_attention`, `mem_efficient` (xFormers), and `math` (naive). We benchmark all three at multiple sequence lengths to see where FlashAttention's IO reduction pays off.

In [ ]:
# --- Experiment 3: Benchmark flash vs math vs mem_efficient backends ---from torch.nn.attention import SDPBackend, sdpa_kerneldef bench_sdpa_backend(backend_ctx, q_bench, k_bench, v_bench, warmup=5, repeats=20):    """Benchmark a specific SDPA backend. Returns median time in ms."""    with backend_ctx:        # Warmup        for _ in range(warmup):            _ = torch.nn.functional.scaled_dot_product_attention(q_bench, k_bench, v_bench, is_causal=True)        torch.cuda.synchronize()        # Timed runs        times_bench = []        for _ in range(repeats):            torch.cuda.synchronize()            t0 = time.perf_counter()            _ = torch.nn.functional.scaled_dot_product_attention(q_bench, k_bench, v_bench, is_causal=True)            torch.cuda.synchronize()            times_bench.append((time.perf_counter() - t0) * 1000)    return sorted(times_bench)[len(times_bench) // 2]  # Medianbench_seq_lengths = [256, 512, 1024, 2048, 4096]batch_bench, heads_bench, dim_bench = 2, 32, 128results_bench = {"flash": [], "mem_efficient": [], "math": []}backends_map = {    "flash": sdpa_kernel(SDPBackend.FLASH_ATTENTION),    "mem_efficient": sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION),    "math": sdpa_kernel(SDPBackend.MATH),}for seq_bench in bench_seq_lengths:    q_b = torch.randn(batch_bench, heads_bench, seq_bench, dim_bench, device=device, dtype=torch.float16)    k_b = torch.randn(batch_bench, heads_bench, seq_bench, dim_bench, device=device, dtype=torch.float16)    v_b = torch.randn(batch_bench, heads_bench, seq_bench, dim_bench, device=device, dtype=torch.float16)    for name, ctx in backends_map.items():        try:            t_ms = bench_sdpa_backend(ctx, q_b, k_b, v_b)            results_bench[name].append(t_ms)            print(f"N={seq_bench:>5d} | {name:>14s}: {t_ms:.2f} ms")        except RuntimeError as e:            results_bench[name].append(None)            print(f"N={seq_bench:>5d} | {name:>14s}: UNSUPPORTED ({e})")    del q_b, k_b, v_b    torch.cuda.empty_cache()# Plot comparisonfig_bench, ax_bench = plt.subplots(figsize=(9, 5))for name, times_list in results_bench.items():    valid = [(s, t) for s, t in zip(bench_seq_lengths, times_list) if t is not None]    if valid:        ax_bench.plot([v[0] for v in valid], [v[1] for v in valid], 'o-', label=name, linewidth=2)ax_bench.set_xlabel("Sequence Length")ax_bench.set_ylabel("Latency (ms)")ax_bench.set_title("SDPA Backend Comparison (B=2, H=32, d=128, causal)")ax_bench.legend()ax_bench.grid(True, alpha=0.3)ax_bench.set_yscale("log")plt.tight_layout()plt.show()

## Experiment 4: Mistral-7B Eager vs FlashAttention-2We load Mistral-7B-v0.1 with both `eager` (standard) and `flash_attention_2` attention implementations, then compare prefill latency and peak memory at a realistic prompt length.

In [ ]:
# --- Experiment 4: Mistral-7B eager vs flash_attention_2 ---from transformers import AutoModelForCausalLM, AutoTokenizerimport osos.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"model_name = "mistralai/Mistral-7B-v0.1"tokenizer_m7b = AutoTokenizer.from_pretrained(model_name, use_fast=True)# Test prompt: ~512 tokensprompt_text = "The transformer architecture " * 128input_ids_m7b = tokenizer_m7b(prompt_text, return_tensors="pt", truncation=True, max_length=512).input_ids.to(device)actual_seq_len = input_ids_m7b.shape[1]print(f"Input sequence length: {actual_seq_len} tokens")def measure_prefill(attn_impl):    """Load model with given attn implementation, measure prefill latency and memory."""    torch.cuda.empty_cache()    torch.cuda.reset_peak_memory_stats()    model_m7b = AutoModelForCausalLM.from_pretrained(        model_name,        attn_implementation=attn_impl,        dtype=torch.float16,        device_map="auto",    )    model_m7b.eval()    # Warmup    with torch.no_grad():        _ = model_m7b(input_ids_m7b)    torch.cuda.synchronize()    # Timed prefill (5 runs, take median)    latencies_m7b = []    for _ in range(5):        torch.cuda.synchronize()        t_start = time.perf_counter()        with torch.no_grad():            _ = model_m7b(input_ids_m7b)        torch.cuda.synchronize()        latencies_m7b.append((time.perf_counter() - t_start) * 1000)    peak_mem_gb = torch.cuda.max_memory_allocated() / 1e9    median_lat = sorted(latencies_m7b)[2]    del model_m7b    torch.cuda.empty_cache()    return median_lat, peak_mem_gb# Measure both implementationsprint("\nLoading with eager attention...")eager_lat, eager_mem = measure_prefill("eager")print(f"  Latency: {eager_lat:.1f} ms | Peak Memory: {eager_mem:.2f} GB")print("\nLoading with flash_attention_2...")flash_lat, flash_mem = measure_prefill("flash_attention_2")print(f"  Latency: {flash_lat:.1f} ms | Peak Memory: {flash_mem:.2f} GB")# Summaryspeedup_pct = (eager_lat - flash_lat) / eager_lat * 100mem_saving_pct = (eager_mem - flash_mem) / eager_mem * 100print(f"\n--- Results at {actual_seq_len} tokens ---")print(f"Latency reduction: {speedup_pct:.1f}%")print(f"Memory reduction:  {mem_saving_pct:.1f}%")

In [ ]:
# --- Visualize Experiment 4 results ---fig_m7b, (ax_lat, ax_mem) = plt.subplots(1, 2, figsize=(10, 4))# Latency comparisonimplementations = ["eager", "flash_attention_2"]latencies_plot = [eager_lat, flash_lat]colors_plot = ["#fef3c7", "#dcfce7"]ax_lat.bar(implementations, latencies_plot, color=colors_plot, edgecolor="#000", linewidth=1.5)ax_lat.set_ylabel("Prefill Latency (ms)")ax_lat.set_title(f"Mistral-7B Prefill @ {actual_seq_len} tokens")ax_lat.grid(True, alpha=0.3, axis='y')# Memory comparisonmemories_plot = [eager_mem, flash_mem]ax_mem.bar(implementations, memories_plot, color=colors_plot, edgecolor="#000", linewidth=1.5)ax_mem.set_ylabel("Peak GPU Memory (GB)")ax_mem.set_title("Peak Memory Usage")ax_mem.grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()print(f"\nFlashAttention-2 saves {speedup_pct:.0f}% latency and {mem_saving_pct:.0f}% memory on Mistral-7B prefill.")

## Key Takeaways1. **Standard attention is memory-bound**, not compute-bound. The N×N score matrix dominates HBM traffic.2. **Tiling + online softmax** eliminates the quadratic memory term while producing exact results.3. **PyTorch SDPA** (`scaled_dot_product_attention`) is the easiest way to use FlashAttention. It selects the best backend automatically.4. **FlashAttention helps most during prefill** with long sequences. During decode (N_q=1), there is no N² matrix, so FlashDecoding or PagedAttention kernels are more relevant.5. **FA-2 and FA-3** improve parallelism and exploit newer hardware (H100 TMA, FP8), but the core insight remains the same: keep data in SRAM.